<a href="https://colab.research.google.com/github/shin584/project/blob/1D_models/models/Cas12a_model_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, BatchNormalization,
    Dropout, Bidirectional, LSTM, GlobalAveragePooling1D, LeakyReLU
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os
from google.colab import drive

# 1. 구글 드라이브 마운트 및 경로 설정
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/project_shared/data'
file_name = 'kim_2018_cas12a_dataset.xlsx'
file_path = os.path.join(base_path, file_name)

# 경로 내 파일 확인
if os.path.exists(base_path):
    print(f"폴더 확인됨: {base_path}")
    print(f"폴더 내 파일 목록: {os.listdir(base_path)}")
else:
    print(f"경고: {base_path} 폴더를 찾을 수 없습니다.")

if os.path.exists(file_path):
    print(f"파일을 찾았습니다! 최종 경로: {file_path}")
else:
    print(f"파일을 찾을 수 없습니다: {file_path}")

def one_hot_encode(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    sequence = sequence.upper()
    return np.array([mapping.get(base, [0,0,0,0]) for base in sequence])

def extract_seq_features(seq):
    protospacer = seq[8:31]
    gc_total = (protospacer.count('G') + protospacer.count('C')) / 23
    gc_seed  = (protospacer[-8:].count('G') + protospacer[-8:].count('C')) / 8
    dinuc_map = {a+b: i for i, (a, b) in enumerate([(x, y) for x in 'ACGT' for y in 'ACGT'])}
    dinuc_features = np.zeros(22 * 16, dtype=np.float32)
    for i in range(22):
        pair = protospacer[i:i+2]
        if pair in dinuc_map:
            dinuc_features[i * 16 + dinuc_map[pair]] = 1.0
    return np.array([gc_total, gc_seed], dtype=np.float32), dinuc_features

def load_cas12a_dataset(filepath):
    print("Cas12a 데이터 로딩 중...")
    if filepath.endswith(('.csv', '.txt')):
        df = pd.read_csv(filepath)
    elif filepath.endswith('.tsv'):
        df = pd.read_csv(filepath, sep='\t')
    elif filepath.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(filepath)
    else:
        raise ValueError("Unsupported file format.")
    seq_col = '34 bp synthetic target and target context sequence\n(4 bp + PAM + 23 bp protospacer + 3 bp)'
    target_col = 'Indel freqeuncy\n(Background substracted, %)'
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
    df = df.dropna(subset=[seq_col, target_col])
    X_seq, X_feat, y = [], [], []
    for _, row in df.iterrows():
        seq = str(row[seq_col]).strip().upper()
        if len(seq) > 34: seq = seq[:34]
        elif len(seq) < 34: seq = seq + ('N' * (34 - len(seq)))
        gc_feat, dinuc_feat = extract_seq_features(seq)
        X_seq.append(one_hot_encode(seq))
        X_feat.append(np.concatenate([gc_feat, dinuc_feat]))
        y.append(float(row[target_col]) / 100.0)
    return np.array(X_seq, dtype=np.float32), np.array(X_feat, dtype=np.float32), np.clip(np.array(y, dtype=np.float32), 0, 1)

def create_cas12a_model():
    seq_input = Input(shape=(34, 4), name='Input_Seq')
    x1 = Conv1D(64, kernel_size=4, padding='same')(seq_input)
    x1 = BatchNormalization()(x1)
    x1 = LeakyReLU(negative_slope=0.1)(x1)
    x2 = Conv1D(64, kernel_size=8, padding='same')(seq_input)
    x2 = BatchNormalization()(x2)
    x2 = LeakyReLU(negative_slope=0.1)(x2)
    x3 = Conv1D(64, kernel_size=15, padding='same')(seq_input)
    x3 = BatchNormalization()(x3)
    x3 = LeakyReLU(negative_slope=0.1)(x3)
    x = tf.keras.layers.Concatenate()([x1, x2, x3])
    x = Dropout(0.4)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    x = Dropout(0.4)(x)
    x = GlobalAveragePooling1D()(x)
    feat_input = Input(shape=(354,), name='Input_Feat')
    f = Dense(64, activation='relu')(feat_input)
    f = Dropout(0.3)(f)
    x = tf.keras.layers.Concatenate()([x, f])
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=[seq_input, bean_input] if 'bean_input' in locals() else [seq_input, feat_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])
    return model

def train_5fold(X_seq, X_feat, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_results, fold_models = [], []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_seq)):
        print(f"\n--- Fold {fold+1} Training ---")
        model = create_cas12a_model()
        # verbose=1로 변경하여 학습 과정을 출력합니다.
        model.fit([X_seq[train_idx], X_feat[train_idx]], y[train_idx],
                  validation_data=([X_seq[val_idx], X_feat[val_idx]], y[val_idx]),
                  epochs=5, batch_size=32, verbose=1)
        fold_models.append(model)
    return fold_models, fold_results

if __name__ == '__main__':
    if os.path.exists(file_path):
        X_seq, X_feat, y = load_cas12a_dataset(file_path)
        fold_models, fold_results = train_5fold(X_seq, X_feat, y)
        print("\n학습 완료!")
    else:
        print("파일이 없어 학습을 진행할 수 없습니다.")

Mounted at /content/drive
폴더 확인됨: /content/drive/MyDrive/project_shared/data
폴더 내 파일 목록: ['kim_2020_table_s8.xlsx', 'Supplementary_Table_1_Saureus_model_input.csv', 'kim_2018_cas12a_dataset.xlsx', 'SaCas9_model_pth']
파일을 찾았습니다! 최종 경로: /content/drive/MyDrive/project_shared/data/kim_2018_cas12a_dataset.xlsx
Cas12a 데이터 로딩 중...

--- Fold 1 Training ---
Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - loss: 0.0804 - mae: 0.2375 - val_loss: 0.0710 - val_mae: 0.2064
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - loss: 0.0556 - mae: 0.1855 - val_loss: 0.0481 - val_mae: 0.1654
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0479 - mae: 0.1683 - val_loss: 0.0465 - val_mae: 0.1606
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0446 - mae: 0.1612 - val_loss: 0.0458 - val_mae: 0.1596
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.0423 - mae: 0.1558 - val_loss: 0.0455 - val_mae: 0.1590

--- Fold 2 Training ---
Epoch 1/5
375/375 ━━━━━━━━━━━━━━━